# Airthings API Exploration

Exploring Airthings API authentication, device metadata, and air quality measurements using Python.

In [7]:
import json
import os
from datetime import datetime
from pathlib import Path

import requests
from dotenv import load_dotenv


load_dotenv()

CLIENT_ID = os.getenv("AIRTHINGS_CLIENT_ID")
CLIENT_SECRET = os.getenv("AIRTHINGS_CLIENT_SECRET")

TOKEN_URL = "https://accounts-api.airthings.com/v1/token"
DEVICES_URL = "https://ext-api.airthings.com/v1/devices"

RAW_DATA_DIR = Path("data/raw")


def get_access_token():
    payload = {
        "grant_type": "client_credentials",
        "scope": "read:device:current_values",
    }

    response = requests.post(
        TOKEN_URL,
        data=payload,
        auth=(CLIENT_ID, CLIENT_SECRET),
    )

    response.raise_for_status()
    return response.json()["access_token"]


def get_devices(access_token):
    headers = {"Authorization": f"Bearer {access_token}"}

    response = requests.get(
        DEVICES_URL,
        headers=headers,
    )

    response.raise_for_status()
    return response.json()


def save_json(data, filename_prefix):
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = RAW_DATA_DIR / f"{filename_prefix}_{timestamp}.json"

    with open(file_path, "w") as file:
        json.dump(data, file, indent=2)

    return file_path


def get_current_values(access_token, device_id):
    headers = {"Authorization": f"Bearer {access_token}"}

    url = f"https://ext-api.airthings.com/v1/devices/{device_id}/latest-samples"

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return response.json()

def c_to_f(celsius):
    return (celsius * 9 / 5) + 32


def bq_to_pci(bq):
    return bq / 37


def format_airthings_row(sample_json):
    data = sample_json["data"]

    timestamp = datetime.fromtimestamp(data["time"])

    return {
        "time_local": timestamp.strftime("%Y-%m-%d %I:%M:%S %p"),
        "temp_f": round(c_to_f(data["temp"]), 1),
        "humidity_percent": data.get("humidity"),
        "co2_ppm": data.get("co2"),
        "voc_ppb": data.get("voc"),
        "pm25": data.get("pm25"),
        "pm1": data.get("pm1"),
        "radon_bq_m3": data.get("radonShortTermAvg"),
        "radon_pci_l": round(bq_to_pci(data["radonShortTermAvg"]), 2),
        "pressure_hpa": data.get("pressure"),
        "battery_percent": data.get("battery"),
    }


def print_simple_table(row):
    for key, value in row.items():
        print(f"{key:20} {value}")

In [8]:
if __name__ == "__main__":
    token = get_access_token()
    devices = get_devices(token)

    device_id = devices["devices"][0]["id"]
    current_values = get_current_values(token, device_id)

    saved_path = save_json(current_values, "latest_samples")

    row = format_airthings_row(current_values)

    print("Success!")
    print(f"Saved response to: {saved_path}")
    print()
    print_simple_table(row)

Success!
Saved response to: data/raw/latest_samples_20260602_185952.json

time_local           2026-06-02 06:58:59 PM
temp_f               74.8
humidity_percent     52.0
co2_ppm              462.0
voc_ppb              198.0
pm25                 10.0
pm1                  7.0
radon_bq_m3          8.0
radon_pci_l          0.22
pressure_hpa         1003.0
battery_percent      100


In [9]:
import json
from pathlib import Path
import pandas as pd

raw_path = Path("../data/raw")

list(raw_path.glob("*"))

[PosixPath('../data/raw/.gitkeep')]

In [11]:
pd.json_normalize(devices["devices"])

,id,deviceType,sensors,productName,segment.id,segment.name,segment.started,segment.active,location.id,location.name
0,2960154462,VIEW_PLUS,"[radonShortTermAvg, temp, humidity, pressure, ...",View Plus,ec023d06-b965-4f9c-b165-02f46b195d0a,Bedroom - North wall,2026-04-11T23:17:07,True,6d4b48d4-39a2-4a94-bfed-8b33a047b9c5,My Home
